# Pruebas de idempotencia: foto del estado del pipeline

Toma una **foto** de las tablas de datos (Bronze, Silver, Gold y snapshots de KPIs) y la guarda en `metadata_dataco.pruebas_calidad.hd_foto_idempotencia`. Se corre **antes y después** de cada corrida del Workflow para comprobar que repetir o solapar una ventana no duplica ni altera datos.

**Cómo se usa**
1. Antes de la prueba: `etiqueta` = `antes_B` (sin `etiqueta_comparar`).
2. Corre el Workflow con la ventana de la prueba.
3. Después: `etiqueta` = `despues_B`, `etiqueta_comparar` = `antes_B`. La última celda muestra la comparación.

**Qué esperar**

| Prueba | Ventana | Métricas de datos | Cambia |
|---|---|---|---|
| A. Repetir un corte | 22→28 | Todas iguales | Solo la traza (`hd_etl_log`, `hd_ejecucion_control`) |
| B. Ventana solapada | 25→31 | Bronze pasa a tener todas las filas del archivo (`bronze_filas` = `bronze_filas_archivo`), con 0 duplicados. Sin versiones SCD2 nuevas en pedidos antiguos. Un snapshot nuevo (01-31) con `es_ultimo_snapshot` en él. | Bronze, Silver, Gold (solo pedidos nuevos), snapshots |
| C. Repetir una ventana vieja | 22→28 | Todas iguales respecto a `despues_B` | Solo la traza |

Las métricas de tipo `TRAZA` crecen siempre (cada corrida deja su registro); las de tipo `DATOS` son las que prueban idempotencia.

In [0]:
dbutils.widgets.text("etiqueta", "despues_B", "Etiqueta de esta foto")
dbutils.widgets.text("etiqueta_comparar", "", "Etiqueta con la que comparar (opcional)")

ETIQUETA = dbutils.widgets.get("etiqueta").strip()
ETIQUETA_COMPARAR = dbutils.widgets.get("etiqueta_comparar").strip()

if not ETIQUETA:
    raise ValueError("Indica una etiqueta para la foto.")

BRONZE = "brz_dataco.sharepoint.raw_supply_chain"
SILVER = "slv_dataco.supply_chain"
GOLD = "gld_dataco.supply_chain"
CONTROL = "metadata_dataco.supply_chain"
TABLA_FOTO = "metadata_dataco.pruebas_calidad.hd_foto_idempotencia"

print(f"Foto: {ETIQUETA}" + (f" | comparar con: {ETIQUETA_COMPARAR}" if ETIQUETA_COMPARAR else ""))

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS metadata_dataco.pruebas_calidad")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLA_FOTO} (
    etiqueta   STRING    COMMENT 'Nombre de la foto, por ejemplo despues_B o despues_B.',
    fecha_foto TIMESTAMP COMMENT 'Momento en que se tomó la foto.',
    metrica    STRING    COMMENT 'Nombre de la métrica medida.',
    tipo       STRING    COMMENT 'DATOS: debe mantenerse en las pruebas de idempotencia. TRAZA: crece con cada corrida.',
    valor      BIGINT    COMMENT 'Valor numérico de la métrica (nulo si no se pudo medir).',
    detalle    STRING    COMMENT 'Texto de apoyo (fechas, cortes) o el error si la métrica falló.'
)
COMMENT 'Fotos del estado del pipeline para las pruebas de idempotencia (antes y después de cada corrida).'
""")

# Repetir la misma etiqueta reemplaza la foto anterior.
spark.sql(f"DELETE FROM {TABLA_FOTO} WHERE etiqueta = '{ETIQUETA}'")

In [0]:
metricas = []   # (nombre, tipo, sql). La consulta devuelve valor y, opcionalmente, un detalle.

def agregar(nombre, tipo, sql):
    metricas.append((nombre, tipo, sql))

# ---------------- Bronze ----------------
agregar("bronze_filas", "DATOS", f"SELECT COUNT(*) FROM {BRONZE}")
agregar("bronze_filas_archivo", "DATOS", f"SELECT MAX(filas_archivo) FROM {BRONZE}")
agregar("bronze_duplicados_por_linea", "DATOS", f"""
    SELECT COALESCE(SUM(c - 1), 0) FROM (
        SELECT COUNT(*) AS c FROM {BRONZE} GROUP BY order_id, order_item_id HAVING COUNT(*) > 1)""")
agregar("bronze_dias_cargados", "DATOS",
    f"SELECT COUNT(DISTINCT fecha_pedido_carga), CAST(MAX(fecha_pedido_carga) AS STRING) FROM {BRONZE}")
agregar("bronze_cortes", "DATOS", f"""
    SELECT COUNT(*), CONCAT_WS(' | ', SORT_ARRAY(COLLECT_LIST(
        CONCAT(CAST(fecha_inicio_corte AS STRING), '->', CAST(fecha_fin_corte AS STRING), ':', CAST(n AS STRING)))))
    FROM (SELECT fecha_inicio_corte, fecha_fin_corte, COUNT(*) AS n FROM {BRONZE} GROUP BY 1, 2)""")

# ---------------- Silver ----------------
agregar("silver_pedidos_filas", "DATOS", f"SELECT COUNT(*) FROM {SILVER}.hd_pedidos")
agregar("silver_pedidos_valido", "DATOS", f"SELECT COUNT(*) FROM {SILVER}.hd_pedidos WHERE estado_calidad = 'VALIDO'")
agregar("silver_pedidos_cuarentena", "DATOS", f"SELECT COUNT(*) FROM {SILVER}.hd_pedidos WHERE estado_calidad = 'CUARENTENA'")
agregar("silver_pedidos_ids_repetidos", "DATOS", f"""
    SELECT COUNT(*) FROM (SELECT order_id FROM {SILVER}.hd_pedidos
                          WHERE estado_calidad = 'VALIDO' GROUP BY order_id HAVING COUNT(*) > 1)""")
agregar("silver_pedido_detalle_filas", "DATOS", f"SELECT COUNT(*) FROM {SILVER}.hd_pedido_detalle")
agregar("silver_detalle_ids_repetidos", "DATOS", f"""
    SELECT COUNT(*) FROM (SELECT order_item_id FROM {SILVER}.hd_pedido_detalle
                          WHERE estado_calidad = 'VALIDO' GROUP BY order_item_id HAVING COUNT(*) > 1)""")
agregar("silver_clientes_filas", "DATOS", f"SELECT COUNT(*) FROM {SILVER}.md_clientes")
agregar("silver_productos_filas", "DATOS", f"SELECT COUNT(*) FROM {SILVER}.md_productos")
agregar("silver_ubicaciones_filas", "DATOS", f"SELECT COUNT(*) FROM {SILVER}.md_ubicaciones")

# ---------------- Gold: hechos y dimensiones (SCD2) ----------------
for nombre, tabla in [("orders", "desc_ft_orders"), ("customers", "desc_dim_customers"), ("products", "desc_dim_products")]:
    agregar(f"gold_{nombre}_filas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.{tabla}")
    agregar(f"gold_{nombre}_vigentes", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.{tabla} WHERE es_vigente = true")
    agregar(f"gold_{nombre}_versiones_cerradas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.{tabla} WHERE es_vigente = false")

agregar("gold_orders_vigentes_repetidos", "DATOS", f"""
    SELECT COUNT(*) FROM (SELECT order_id FROM {GOLD}.desc_ft_orders
                          WHERE es_vigente = true GROUP BY order_id HAVING COUNT(*) > 1)""")
agregar("gold_orders_huella_contenido", "DATOS", f"""
    SELECT SUM(PMOD(XXHASH64(order_id, order_status, delivery_status), 1000003))
    FROM {GOLD}.desc_ft_orders WHERE es_vigente = true""")
agregar("gold_order_details_filas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.desc_ft_order_details")
agregar("gold_order_summary_filas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.desc_ft_order_summary")
agregar("gold_locations_filas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.desc_dim_locations")
agregar("gold_fecha_filas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.desc_dim_fecha")
agregar("gold_shipping_mode_filas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.desc_dim_shipping_mode")

# ---------------- Gold: snapshots de KPIs ----------------
for nombre, tabla in [("resumen", "desc_ft_kpi_resumen_ejecutivo"),
                      ("region", "desc_ft_kpi_detalle_region"),
                      ("alertas", "desc_ft_kpi_alertas")]:
    agregar(f"kpi_{nombre}_filas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.{tabla}")
    agregar(f"kpi_{nombre}_snapshots", "DATOS", f"""
        SELECT COUNT(DISTINCT fecha_snapshot),
               CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(CAST(fecha_snapshot AS STRING)))) FROM {GOLD}.{tabla}""")
    agregar(f"kpi_{nombre}_ultimo_snapshot", "DATOS", f"""
        SELECT COUNT(DISTINCT fecha_snapshot),
               CONCAT_WS(' | ', SORT_ARRAY(COLLECT_SET(CAST(fecha_snapshot AS STRING))))
        FROM {GOLD}.{tabla} WHERE es_ultimo_snapshot = true""")

agregar("kpi_detalle_orden_filas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.desc_ft_kpi_detalle_orden")
agregar("kpi_detalle_orden_vigentes", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.desc_ft_kpi_detalle_orden WHERE es_vigente = true")
agregar("kpi_detalle_orden_versiones_cerradas", "DATOS", f"SELECT COUNT(*) FROM {GOLD}.desc_ft_kpi_detalle_orden WHERE es_vigente = false")

# ---------------- Traza (crece con cada corrida; es normal) ----------------
agregar("traza_hd_etl_log_filas", "TRAZA", f"SELECT COUNT(*) FROM {CONTROL}.hd_etl_log")
agregar("traza_hd_ejecucion_control_filas", "TRAZA", f"SELECT COUNT(*) FROM {CONTROL}.hd_ejecucion_control")

print(f"{len(metricas)} métricas definidas")

In [0]:
from datetime import datetime

ahora = datetime.now()
filas = []

for nombre, tipo, sql in metricas:
    try:
        fila = spark.sql(sql).collect()[0]
        valor = int(fila[0]) if fila[0] is not None else None
        detalle = str(fila[1]) if len(fila) > 1 and fila[1] is not None else None
    except Exception as error:
        valor = None
        detalle = "ERROR: " + str(error).replace("\n", " ")[:200]
    filas.append((ETIQUETA, ahora, nombre, tipo, valor, detalle))

df_foto = spark.createDataFrame(
    filas, "etiqueta string, fecha_foto timestamp, metrica string, tipo string, valor bigint, detalle string"
)
df_foto.write.mode("append").saveAsTable(TABLA_FOTO)

con_error = [f[2] for f in filas if f[5] and f[5].startswith("ERROR")]
print(f"✓ Foto '{ETIQUETA}' guardada: {len(filas)} métricas")
if con_error:
    print("⚠️ Métricas que fallaron (revisa nombres de tabla o columna):", ", ".join(con_error))

display(spark.table(TABLA_FOTO).filter(f"etiqueta = '{ETIQUETA}'").orderBy("tipo", "metrica"))

In [0]:
if not ETIQUETA_COMPARAR:
    print("Sin etiqueta para comparar: foto guardada, no hay comparación.")
else:
    hay = spark.table(TABLA_FOTO).filter(f"etiqueta = '{ETIQUETA_COMPARAR}'").count()
    if hay == 0:
        raise ValueError(f"No existe la foto '{ETIQUETA_COMPARAR}'. Toma primero esa foto.")

    comparacion = spark.sql(f"""
        SELECT a.tipo,
               a.metrica,
               b.valor   AS valor_antes,
               a.valor   AS valor_ahora,
               a.valor - b.valor AS diferencia,
               CASE WHEN NOT (a.valor <=> b.valor) OR NOT (a.detalle <=> b.detalle) THEN 'CAMBIO' ELSE 'IGUAL' END AS estado,
               b.detalle AS detalle_antes,
               a.detalle AS detalle_ahora
        FROM {TABLA_FOTO} a
        JOIN {TABLA_FOTO} b ON a.metrica = b.metrica
        WHERE a.etiqueta = '{ETIQUETA}' AND b.etiqueta = '{ETIQUETA_COMPARAR}'
        ORDER BY a.tipo, estado DESC, a.metrica
    """)
    display(comparacion)

    cambios_datos = comparacion.filter("tipo = 'DATOS' AND estado = 'CAMBIO'").count()
    print(f"Métricas de DATOS que cambiaron entre '{ETIQUETA_COMPARAR}' y '{ETIQUETA}': {cambios_datos}")
    print("En las pruebas A y C se espera 0. En la B se espera ver los cambios descritos en el markdown de arriba.")